# Missing Values

Most learning algorithms cannot train on data with gaps, so we either drop incomplete rows (wasteful) or **impute** the missing values from the data we already have.

In this lesson we learn to:

- Identify common causes of missing data
- Drop missing rows and see why it loses information
- Fill gaps with `SimpleImputer` (mean / median)
- Fill gaps with `KNNImputer`, scaling features first so distances are meaningful
- Recover the original units with `inverse_transform`

In [ ]:
# https://numpy.org/doc/stable/
import numpy as np

# https://pandas.pydata.org/docs/
import pandas as pd

# https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
from sklearn.impute import SimpleImputer, KNNImputer

# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
from sklearn.preprocessing import StandardScaler

## 1. Why data goes missing

Most library algorithms require that the data contain no gaps (**missing values**). Data goes missing for many reasons:

- **Human error:** a value is accidentally omitted, or a user declines to answer to protect their privacy.
- **Technical failure:** a sensor malfunctions or a connection drops during data transfer.
- **Data merging:** gaps appear from join operations (e.g. SQL joins) or from changes in table structure over time.
- **Attrition:** data is lost when participants drop out of longitudinal studies or delete their accounts.
    - In clinical trials, a patient may drop out due to migration, death, loss of interest in follow-up, a worsening condition, or severe side effects.
    - In subscription systems, churned customers stop using the service, so their data appears empty in the months after they leave.

## 2. Sample data

- We use a small set of health measurements: age, systolic blood pressure, cholesterol, and body mass index (BMI).
- A few values are knocked out so we can practice filling them in.

In [ ]:
# Reproducible synthetic patient measurements
np.random.seed(42)
n_patients = 12

# Each measurement loosely tracks age, plus random noise
age = np.random.randint(35, 70, size=n_patients).astype(float)
blood_pressure = 100 + 0.5 * age + np.random.randn(n_patients) * 8  # systolic, mmHg
cholesterol = 150 + 0.8 * age + np.random.randn(n_patients) * 15  # mg/dL
bmi = 22 + 0.1 * age + np.random.randn(n_patients) * 2  # body mass index

df = pd.DataFrame(
    {
        "age": age,
        "blood_pressure": blood_pressure,
        "cholesterol": cholesterol,
        "bmi": bmi,
    }
)

# Knock out a few values to mimic real-world gaps
df.loc[1, "blood_pressure"] = np.nan
df.loc[3, "cholesterol"] = np.nan
df.loc[5, "age"] = np.nan
df.loc[7, "bmi"] = np.nan
df.loc[9, ["blood_pressure", "cholesterol"]] = np.nan

df

### Naive solution: drop the rows

The simplest fix is to remove every row that has a missing value with [`dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html).

- This throws away the *other*, perfectly good values in those rows.
- A single missing `age`, for example, discards that patient's blood pressure, cholesterol, and BMI too.

In [ ]:
df_dropped = df.dropna()

print(f"Original rows: {len(df)}")
print(f"Rows left after dropna(): {len(df_dropped)}")
df_dropped

## 3. Imputation methods

A better approach is to **impute** (fill in) the missing values, inferring them from the known part of the data. There are two main families:

- **Univariate** ([`SimpleImputer`](https://scikit-learn.org/stable/modules/impute.html#univariate-feature-imputation)): use only the *same* variable, e.g. its mean, median, most frequent value, or a constant.
- **Multivariate** ([`IterativeImputer`](https://scikit-learn.org/stable/modules/impute.html#multivariate-feature-imputation)): use the *other* variables, treating the gap as a prediction task (regression or classification, depending on the variable's type).

## 4. Simple imputation with `SimpleImputer`

[`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) fills each gap using a single statistic of that column: the **mean**, **median**, most frequent value, or a constant.

We first pull the raw values into a NumPy array, then impute with the mean.

In [ ]:
# Work on the raw values as a NumPy array
X = df.to_numpy()

mean_imputer = SimpleImputer(strategy="mean")  # fill each gap with the column mean
X_mean = mean_imputer.fit_transform(X)

df_mean = pd.DataFrame(X_mean, columns=df.columns, index=df.index)
df_mean.round(2)

The **median** is often a better choice than the mean because it is robust to outliers.

In [ ]:
median_imputer = SimpleImputer(strategy="median")  # fill each gap with the column median
X_median = median_imputer.fit_transform(X)

df_median = pd.DataFrame(X_median, columns=df.columns, index=df.index)
df_median.round(2)

## 5. K-Nearest-Neighbors imputation with `KNNImputer`

[`KNNImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html) fills each missing value using the closest samples by distance (missing entries are ignored when measuring distance).

### k-NN classifier

A **k-NN classifier** outputs a class label: a new point is assigned the majority class among its *k* nearest neighbors.

![Application of a k-NN classifier with k = 3 neighbors](https://upload.wikimedia.org/wikipedia/commons/7/78/KNN_decision_surface_animation.gif)

- **Left:** for the test point "?", the algorithm finds the 3 closest training points and takes the majority vote, classifying it as "red".
- **Right:** repeating the prediction across the whole feature space (X1, X2) traces out the "decision surface".

A k-NN classifier can fill in **categorical** variables.

### k-NN regression

**k-NN regression** (also called k-NN smoothing) is used for **continuous** variables. It predicts a weighted **average of the k nearest neighbors**, weighted by the inverse of their distance.

![Imputation of a continuous value from nearby points](../assets/impute_continuous_value.png)

This lets us fill any missing point in the data.

> **Warning:** Distance-based algorithms like KNN assume the numerical features are on a comparable scale. Skipping this step gives erroneous results, because a large-range feature dominates the distance.

We first scale the features with [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), which shifts each column to a mean of 0 and a standard deviation of 1 (covered in the next lesson).

In [ ]:
scaler = StandardScaler()

# StandardScaler ignores NaNs when learning the mean and standard deviation,
# so we can scale the data before the gaps are filled
X_scaled = scaler.fit_transform(X)

Now we impute on the scaled data.

In [ ]:
knn_imputer = KNNImputer(n_neighbors=3)  # average the 3 closest patients
X_knn = knn_imputer.fit_transform(X_scaled)

Finally we **invert the scaling** with `inverse_transform` to read the filled values back in their original units.

In [ ]:
df_knn = pd.DataFrame(
    scaler.inverse_transform(X_knn),
    columns=df.columns,
    index=df.index,
)
df_knn.round(2)

## Recap

- Dropping rows with `dropna()` is wasteful: one gap discards a whole row of good data.
- `SimpleImputer` fills gaps from a single column statistic (mean / median).
- `KNNImputer` fills gaps from the nearest samples, so **scale first** for meaningful distances.
- `inverse_transform` returns scaled values to their original units.

For more, see [7.4. Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html).